# 14B — Firm-level critical-minerals thematic beta

## Primary question

Do frozen text and holdings-embedding exposure scores predict firms' realized sensitivity to an independently constructed critical-minerals equity-theme factor?

For firm \(i\), vintage \(t\), and daily return \(d\):

\[
R_{i,d}=lpha_{i,t}+eta^M_{i,t}R^{SPY}_d+eta^{CM}_{i,t}CM_d+\epsilon_{i,d}.
\]

The economically relevant outcome is \(eta^{CM}\), not unconditional return. Historical and forward betas are kept separate. This notebook does not load the earlier portfolio-performance results.

## 1. Setup and frozen design

In [1]:
from pathlib import Path
import getpass,hashlib,json,time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
if ROOT.name!='sp500_critical_minerals':
    candidate=Path('/Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/sp500_critical_minerals')
    if candidate.exists():ROOT=candidate
FACTOR=ROOT/'thematic_factor_v1'; OUT=ROOT/'thematic_beta_v1'; RAW=OUT/'raw'; MANIFESTS=OUT/'manifests'
for p in [OUT,RAW,MANIFESTS]:p.mkdir(exist_ok=True)

DESIGN={'version':'firm_thematic_beta_v1','primary_factor':'broad_cm_equity_ex_market',
 'robustness_factors':['producer_ex_market','base_metals_raw','materials_ex_market'],
 'return_frequency':'daily','historical_window_days':252,'forward_target_days':252,
 'common_cutoff':'2025-12-31','minimum_observations':126,
 'time_series_model':'OLS: firm return ~ intercept + SPY return + thematic factor',
 'multiple_share_class_aggregation':'lagged-market-equity weighted PERMNO return within PERMCO',
 'missing_return_policy':'retain CRSP numeric observations only; no imputation',
 'portfolio_results_accessed':False,'scenario_13a_scores_accessed':False}
design_path=MANIFESTS/'thematic_beta_design.json'
if design_path.exists():assert json.loads(design_path.read_text())==DESIGN
else:design_path.write_text(json.dumps(DESIGN,indent=2,sort_keys=True)+'\n')
design_hash=hashlib.sha256(design_path.read_bytes()).hexdigest()
print({'design_sha256':design_hash,'portfolio_results_accessed':False})

{'design_sha256': '1dcb132c2b57988c713a58d859239975ad42af3fb57b60efeef848030e8d3fa9', 'portfolio_results_accessed': False}


## 2. Input audit

Run 14A after its cutoff-alignment revision before continuing.

In [2]:
paths={'factor':FACTOR/'daily_critical_minerals_factors.parquet',
 'windows':FACTOR/'manifests'/'frozen_beta_windows.parquet',
 'signals':ROOT/'long_short_hybrid_v1'/'signed_signal_panel.parquet',
 'bridge':ROOT/'data'/'processed'/'sp500_point_in_time_identifier_bridge.parquet'}
audit=pd.DataFrame([{'input':k,'exists':p.exists(),'sha256':hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None} for k,p in paths.items()])
display(audit);assert audit.exists.all(),'Run corrected 14A or restore a missing frozen input.'
factor=pd.read_parquet(paths['factor']);factor['date']=pd.to_datetime(factor.date)
windows=pd.read_parquet(paths['windows']);signals=pd.read_parquet(paths['signals'])
bridge=pd.read_parquet(paths['bridge'])
assert factor.date.max()<=pd.Timestamp(DESIGN['common_cutoff'])
display(windows)

,input,exists,sha256
0,factor,True,25d1ddb36a98ae97a0f1340d23893e94dfae2fa68189e4...
1,windows,True,dff4de1ec9d33b96c04c0e2875b6f91062ba3ab1750a99...
2,signals,True,0ff8c58effb8f2813265bb45c307ee6d3dc42317327401...
3,bridge,True,8cc2d492e8c86f37be35f9881ece4d6760391679b6996d...


,vintage,signal_cutoff,historical_start,historical_end,historical_days,forward_start,forward_end,forward_days_available,forward_target_days
0,2024q1,2024-03-31,2023-03-29,2024-03-28,252,2024-04-01,2025-04-01,252,252
1,2024q2,2024-06-30,2023-06-29,2024-06-28,252,2024-07-01,2025-07-02,252,252
2,2024q3,2024-09-30,2023-09-29,2024-09-30,252,2024-10-01,2025-10-02,252,252
3,2024q4,2024-12-31,2024-01-02,2024-12-31,252,2025-01-02,2025-12-31,250,252


## 3. Connect to WRDS only when the checkpoint is absent

In [3]:
security_path=RAW/'crsp_daily_security_returns_2023_2025.parquet'
needs_wrds=not security_path.exists()
if needs_wrds:
    import wrds
    username=input('WRDS username: ').strip();password=getpass.getpass('WRDS password: ')
    db=wrds.Connection(wrds_username=username,wrds_password=password,autoconnect=False,verbose=True,
      wrds_connect_args={'sslmode':'require','connect_timeout':30,'application_name':'cm_thematic_beta'})
    db.connect();display(db.raw_sql('select 1 as connection_test'))
else:print('CRSP checkpoint exists; WRDS connection is not required.')

postgresql://hermanthinnd:@wrds-pgdata.wharton.upenn.edu:9737/wrds


,connection_test
0,1


## 4. Acquire the minimal CRSP daily extract

The union contains only securities appearing in one of the four point-in-time S&P 500 universes. Queries are checkpointed as one combined file.

In [4]:
mapping=bridge[['permno','permco']].dropna().drop_duplicates().astype({'permno':int,'permco':int})
assert mapping.groupby('permno').permco.nunique().max()==1
permnos=sorted(mapping.permno.unique())

def rollback():
    try:db.connection.rollback()
    except Exception:pass

if security_path.exists():
    security=pd.read_parquet(security_path);source='local_checkpoint'
else:
    candidates=[
     {'table':'crsp_a_stock.wrds_dsfv2_query','date':'dlycaldt','ret':'dlyret','prc':'dlyprc','shrout':'shrout'},
     {'table':'crsp_a_stock.dsf_v2','date':'dlycaldt','ret':'dlyret','prc':'dlyprc','shrout':'shrout'},
     {'table':'crsp_a_stock.dsf','date':'date','ret':'ret','prc':'prc','shrout':'shrout'},
     {'table':'crsp.dsf','date':'date','ret':'ret','prc':'prc','shrout':'shrout'}]
    spec=None;errors=[]
    for candidate in candidates:
        try:
            db.raw_sql(f"select permno,{candidate['date']},{candidate['ret']},{candidate['prc']},{candidate['shrout']} from {candidate['table']} limit 1")
            spec=candidate;break
        except Exception as error:
            errors.append((candidate['table'],type(error).__name__));rollback()
    if spec is None:raise RuntimeError(f'No daily CRSP table queryable: {errors}')
    frames=[]
    for start in range(0,len(permnos),250):
        ids=','.join(map(str,permnos[start:start+250]))
        q=f"""select permno,{spec['date']} as date,{spec['ret']} as ret,{spec['prc']} as prc,{spec['shrout']} as shrout
        from {spec['table']} where {spec['date']} between '2023-03-01' and '2025-12-31' and permno in ({ids})"""
        frames.append(db.raw_sql(q,date_cols=['date']));print(f'batch {start//250+1}/{int(np.ceil(len(permnos)/250))} complete')
    security=pd.concat(frames,ignore_index=True);security.to_parquet(security_path,index=False);source=spec['table']
print({'source':source,'rows':len(security),'permnos':security.permno.nunique()})

batch 1/3 complete
batch 2/3 complete
batch 3/3 complete
{'source': 'crsp_a_stock.wrds_dsfv2_query', 'rows': 365845, 'permnos': 518}


## 5. Clean and aggregate security returns to firms

In [5]:
security['date']=pd.to_datetime(security.date)
for c in ['permno','ret','prc','shrout']:security[c]=pd.to_numeric(security[c],errors='coerce')
security=security.merge(mapping,on='permno',how='inner',validate='many_to_one')
security=security.sort_values(['permno','date']).drop_duplicates(['permno','date'],keep='last')
security['market_equity']=security.prc.abs()*security.shrout
security['lag_market_equity']=security.groupby('permno').market_equity.shift(1)
valid=security.dropna(subset=['ret']).copy()

def aggregate_firm(g):
    w=g.lag_market_equity.where(g.lag_market_equity.gt(0))
    value=np.average(g.ret,weights=w) if w.notna().all() and w.sum()>0 else g.ret.mean()
    return pd.Series({'firm_return':value,'underlying_permnos':g.permno.nunique()})
firm_daily=(valid.groupby(['permco','date'],as_index=False).apply(aggregate_firm,include_groups=False).reset_index())
if 'level_2' in firm_daily:firm_daily=firm_daily.drop(columns='level_2')
firm_daily.to_parquet(OUT/'crsp_daily_firm_returns.parquet',index=False)
display(pd.DataFrame([{'rows':len(firm_daily),'firms':firm_daily.permco.nunique(),'first_date':firm_daily.date.min(),
                       'last_date':firm_daily.date.max(),'duplicate_firm_dates':int(firm_daily.duplicated(['permco','date']).sum())}]))

,rows,firms,first_date,last_date,duplicate_firm_dates
0,363657,515,2023-03-01,2025-12-31,0


## 6. Estimate historical and forward betas

The same regression is estimated separately for each firm, vintage, window, and frozen factor. No beta is reported with fewer than 126 observations.

In [6]:
factor_names=[DESIGN['primary_factor']]+DESIGN['robustness_factors'];rows=[]
for w in windows.itertuples(index=False):
    ranges={'historical':(pd.Timestamp(w.historical_start),pd.Timestamp(w.historical_end)),
            'forward':(pd.Timestamp(w.forward_start),min(pd.Timestamp(w.forward_end),pd.Timestamp(DESIGN['common_cutoff'])))}
    vintage_firms=set(signals.loc[signals.vintage.eq(w.vintage),'permco'].astype(int))
    for window,(start,end) in ranges.items():
        base=firm_daily[firm_daily.permco.isin(vintage_firms)&firm_daily.date.between(start,end)].merge(factor,on='date',how='inner')
        for permco,g in base.groupby('permco'):
            for factor_name in factor_names:
                z=g[['firm_return','market_return',factor_name]].dropna()
                if len(z)<DESIGN['minimum_observations']:
                    rows.append({'vintage':w.vintage,'window':window,'permco':permco,'factor':factor_name,'observations':len(z),'alpha':np.nan,'market_beta':np.nan,'thematic_beta':np.nan,'r2':np.nan});continue
                X=np.column_stack([np.ones(len(z)),z.market_return,z[factor_name]])
                coef=np.linalg.lstsq(X,z.firm_return.to_numpy(),rcond=None)[0];pred=X@coef
                sst=((z.firm_return-z.firm_return.mean())**2).sum();sse=((z.firm_return-pred)**2).sum()
                rows.append({'vintage':w.vintage,'window':window,'permco':permco,'factor':factor_name,'observations':len(z),
                             'alpha':coef[0],'market_beta':coef[1],'thematic_beta':coef[2],'r2':1-sse/sst if sst>0 else np.nan})
betas=pd.DataFrame(rows);betas.to_parquet(OUT/'firm_thematic_betas.parquet',index=False)
coverage=(betas.groupby(['vintage','window','factor'],as_index=False).agg(firms=('permco','nunique'),
          estimated_betas=('thematic_beta','count'),median_observations=('observations','median'),median_r2=('r2','median')))
display(coverage);assert betas[betas.factor.eq(DESIGN['primary_factor'])].groupby(['vintage','window']).thematic_beta.count().min()>=350

,vintage,window,factor,firms,estimated_betas,median_observations,median_r2
0,2024q1,forward,base_metals_raw,500,498,252.0,0.133394
1,2024q1,forward,broad_cm_equity_ex_market,500,498,252.0,0.141325
2,2024q1,forward,materials_ex_market,500,498,252.0,0.143143
3,2024q1,forward,producer_ex_market,500,498,252.0,0.141919
4,2024q1,historical,base_metals_raw,500,499,252.0,0.187251
5,2024q1,historical,broad_cm_equity_ex_market,500,499,252.0,0.206079
6,2024q1,historical,materials_ex_market,500,499,252.0,0.219769
7,2024q1,historical,producer_ex_market,500,499,252.0,0.204883
8,2024q2,forward,base_metals_raw,500,497,252.0,0.297198
9,2024q2,forward,broad_cm_equity_ex_market,500,497,252.0,0.298575


## 7. Blinded descriptive diagnostics

These diagnostics inspect beta estimation quality only. Exposure scores are not joined until 14C.

In [7]:
primary=betas[betas.factor.eq(DESIGN['primary_factor'])]
display(primary.groupby(['vintage','window']).thematic_beta.describe(percentiles=[.01,.1,.5,.9,.99]))
stability=(primary.pivot_table(index=['vintage','permco'],columns='window',values='thematic_beta').dropna())
print('Historical-forward beta correlation:',stability.historical.corr(stability.forward))
gates=pd.DataFrame([
 {'gate':'common_cutoff_respected','value':int(firm_daily.date.max()<=pd.Timestamp(DESIGN['common_cutoff'])),'threshold':1,'passed':firm_daily.date.max()<=pd.Timestamp(DESIGN['common_cutoff'])},
 {'gate':'minimum_350_primary_betas_each_cell','value':int(primary.groupby(['vintage','window']).thematic_beta.count().min()),'threshold':350,'passed':primary.groupby(['vintage','window']).thematic_beta.count().min()>=350},
 {'gate':'portfolio_results_unopened','value':0,'threshold':0,'passed':True}])
display(gates);gates.to_parquet(MANIFESTS/'beta_estimation_gates.parquet',index=False);assert gates.passed.all()

count      mean       std       min        1%       10%  \
vintage window                                                                
2024q1  forward     498.0  0.053625  0.143253 -0.240163 -0.177236 -0.088524   
        historical  499.0  0.125999  0.182741 -0.568102 -0.290526 -0.074771   
2024q2  forward     497.0  0.044295  0.150688 -0.318257 -0.180292 -0.097935   
        historical  498.0  0.110478  0.172442 -0.579250 -0.232737 -0.079285   
2024q3  forward     499.0  0.055521  0.144149 -0.349353 -0.201187 -0.089716   
        historical  498.0  0.098504  0.160307 -0.571995 -0.211101 -0.071315   
2024q4  forward     498.0  0.034563  0.114144 -0.228695 -0.136362 -0.073253   
        historical  499.0  0.061363  0.139385 -0.264316 -0.196406 -0.086237   

                         50%       90%       99%       max  
vintage window                                              
2024q1  forward     0.020196  0.221566  0.504616  1.150081  
        historical  0.122707  0.342055  0.587393  1.526290  
2024q2  forward     0.016835  0.209713  0.568012  1.150059  
        historical  0.105477  0.296682  0.609793  1.421421  
2024q3  forward     0.039011  0.222109  0.539558  1.248579  
        historical  0.091164  0.280649  0.554413  1.440984  
2024q4  forward     0.020488  0.148805  0.357069  1.268630  
        historical  0.042250  0.209924  0.434858  1.322462

Historical-forward beta correlation: 0.5769650936620542


,gate,value,threshold,passed
0,common_cutoff_respected,1,1,True
1,minimum_350_primary_betas_each_cell,497,350,True
2,portfolio_results_unopened,0,0,True


## 8. Manifest and handoff

In [8]:
artifacts=[design_path,security_path,OUT/'crsp_daily_firm_returns.parquet',OUT/'firm_thematic_betas.parquet',MANIFESTS/'beta_estimation_gates.parquet']
inventory=pd.DataFrame([{'artifact':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in artifacts])
inventory.to_parquet(MANIFESTS/'artifact_inventory.parquet',index=False)
manifest={'stage':'firm_thematic_beta_estimation_v1','design_sha256':design_hash,'primary_factor':DESIGN['primary_factor'],
 'beta_rows':len(betas),'primary_estimated':int(betas.loc[betas.factor.eq(DESIGN['primary_factor']),'thematic_beta'].count()),
 'common_cutoff':DESIGN['common_cutoff'],'portfolio_results_accessed':False,'exposure_scores_joined':False}
(MANIFESTS/'manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
display(inventory);print('Saved:',OUT)
print('Next: 14C joins frozen scores only after these beta estimates are checkpointed.')

,artifact,bytes,sha256
0,thematic_beta_v1/manifests/thematic_beta_desig...,702,1dcb132c2b57988c713a58d859239975ad42af3fb57b60...
1,thematic_beta_v1/raw/crsp_daily_security_retur...,2597782,283b9c23dc9c8c5619b039c661c4560385268cfdb32b44...
2,thematic_beta_v1/crsp_daily_firm_returns.parquet,3446166,947695ca321f5efb2d0021161575155bc69bdc103b95c6...
3,thematic_beta_v1/firm_thematic_betas.parquet,634209,51ae816c4c0df0b163310eceb1a3b008881dbd6ff37f90...
4,thematic_beta_v1/manifests/beta_estimation_gat...,2834,6f309b8c75aef2c34bf539b8875cbbfd90ea62f7d8bed5...


Saved: /Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/sp500_critical_minerals/thematic_beta_v1
Next: 14C joins frozen scores only after these beta estimates are checkpointed.
